# Trail Surface ID (Video) — YOLO26 mask + CLIP open-vocab

Same workflow as the photo notebook, but operates on an **MP4** video. For each frame:

1. YOLO26 instance segmentation → pick the best trail-class mask
2. Dilate the mask by `MASK_DILATE_PX` so CLIP gets a little context
3. Black out non-trail pixels, crop to mask bbox
4. CLIP open-vocab surface classification
5. Record per-frame results to a CSV; write an annotated `output.mp4`

Per-frame predictions will be jittery — that's expected. The next iteration will smooth with a running average across frames.

## Step 1: Install & import

In [ ]:
!pip install -q ultralytics transformers pillow

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
from IPython.display import display
from google.colab import files
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 2: Upload YOLO26 weights & load both models

In [ ]:
if 'yolo_model' not in locals():
    print("Upload YOLO26 .pt weights (instance segmentation model):")
    yolo_upload = files.upload()
    yolo_weights_path = next(k for k in yolo_upload if k.endswith('.pt'))
    print(f"\nLoading YOLO from {yolo_weights_path}...")
    yolo_model = YOLO(yolo_weights_path)
    print(f"YOLO classes: {yolo_model.names}")

if 'clip_model' not in locals():
    print("\nLoading CLIP...")
    clip_model_name = "openai/clip-vit-base-patch32"
    clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
    clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
    print("CLIP loaded.")

## Step 3: Define surface labels for CLIP

In [ ]:
surface_categories = {
    "Gravel": [
        "a photo of a gravel trail",
        "a photo of a trail made of loose, small crushed rocks",
        "a photo of a crushed stone path"
    ],
    "Paved": [
        "a photo of a paved asphalt path",
        "a photo of a smooth concrete trail",
        "an artificial sidewalk with clean, straight edges",
        "a smooth surface designed for bicycle tires and car wheels"
    ],
    "Boardwalk": [
        "a photo of a wide wooden boardwalk",
        "a photo of a flat, accessible path built with multiple wooden planks",
        "a photo of a wooden bridge wide enough for a wheelchair"
    ],
    "Bog Bridge": [
        "a photo of a narrow bog bridge",
        "a photo of a single wooden log placed on the trail to walk on",
        "a photo of a narrow wooden plank over mud, difficult for a wheelchair"
    ],
    "Rocky": [
        "a photo of a rocky hiking trail",
        "a photo of uneven ground covered in large boulders and stones"
    ],
    "Dirt": [
        "a photo of a packed dirt trail",
        "a photo of a soil footpath through the woods",
        "a photo of a smooth earthen path"
    ],
    "Woodchips": [
        "a photo of woodchips on a trail",
        "a photo of a path covered in shredded bark and mulch"
    ],
    "Grass": [
        "a photo of a grassy footpath",
        "a photo of a mown grass trail through a field"
    ]
}

all_phrases = []
phrase_to_category = {}
for category, phrases in surface_categories.items():
    for phrase in phrases:
        all_phrases.append(phrase)
        phrase_to_category[phrase] = category

print(f"{len(surface_categories)} categories, {len(all_phrases)} prompts.")

## Step 4: Helper functions

In [ ]:
TRAIL_CLASS_HINTS    = ["trail", "path", "sidewalk", "road"]
YOLO_TRAIL_MIN_CONF  = 0.25
MASK_DILATE_PX       = 20  # grow the trail mask outward this many pixels for context


def is_trail_class(class_name):
    cn = class_name.lower()
    return any(h in cn for h in TRAIL_CLASS_HINTS)


def get_best_trail_mask(yolo_result):
    """Pick the trail-class instance with the largest area * confidence."""
    if yolo_result.masks is None or yolo_result.boxes is None or len(yolo_result.boxes) == 0:
        return None, None, None, 0
    masks_np = yolo_result.masks.data.cpu().numpy().astype(bool)
    confs    = yolo_result.boxes.conf.cpu().numpy()
    clss     = yolo_result.boxes.cls.cpu().numpy().astype(int)
    names    = yolo_result.names

    best = None
    for i in range(len(confs)):
        cname = names.get(int(clss[i]), str(int(clss[i])))
        if not is_trail_class(cname) or confs[i] < YOLO_TRAIL_MIN_CONF:
            continue
        area = int(masks_np[i].sum())
        score = area * float(confs[i])
        if best is None or score > best[0]:
            best = (score, masks_np[i], cname, float(confs[i]), area)
    if best is None:
        return None, None, None, 0
    _, mask, cname, conf, area = best
    return mask, cname, conf, area


def dilate_mask(mask_bool, target_hw, px=MASK_DILATE_PX):
    """Resize mask to target (H, W) and dilate by `px` pixels."""
    H, W = target_hw
    if mask_bool.shape != (H, W):
        mask_bool = cv2.resize(mask_bool.astype(np.uint8), (W, H),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
    if px > 0:
        k = 2 * px + 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        mask_bool = cv2.dilate(mask_bool.astype(np.uint8), kernel, iterations=1).astype(bool)
    return mask_bool


def masked_crop_for_clip(pil_img, mask_bool):
    img_np = np.array(pil_img)
    masked = img_np.copy()
    masked[~mask_bool] = 0
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    return Image.fromarray(masked[y1:y2 + 1, x1:x2 + 1])


def fallback_bottom_center_crop(pil_img):
    w, h = pil_img.size
    return pil_img.crop((int(w * 0.2), int(h * 0.6), int(w * 0.8), h))


def clip_classify_surface(pil_img):
    inputs = clip_processor(text=all_phrases, images=pil_img,
                            return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]
    category_scores = {cat: 0.0 for cat in surface_categories}
    for phrase, score in zip(all_phrases, probs):
        cat = phrase_to_category[phrase]
        if score > category_scores[cat]:
            category_scores[cat] = float(score)
    predicted = max(category_scores, key=category_scores.get)
    return predicted, category_scores[predicted] * 100, category_scores


def overlay_mask_bgr(frame_bgr, mask_bool, color_bgr=(128, 255, 0), alpha=0.45):
    """Draw a semi-transparent mask overlay directly on a BGR frame (in-place safe)."""
    out = frame_bgr.copy()
    out[mask_bool] = (alpha * np.array(color_bgr) + (1 - alpha) * out[mask_bool]).astype(np.uint8)
    return out

## Step 5: Upload video

In [ ]:
print("Upload an .mp4 video:")
video_upload = files.upload()
video_path = next(k for k in video_upload if k.lower().endswith('.mp4'))
print(f"\nVideo: {video_path}")

## Step 6: Process video — per-frame YOLO mask + CLIP surface

Writes:
- `output.mp4` — every frame with the dilated trail mask overlay + CLIP label burned in
- `surface_per_frame.csv` — one row per frame with the per-category CLIP scores

In [ ]:
PROGRESS_EVERY = 25  # print a progress line every N frames

cap = cv2.VideoCapture(video_path)
fps         = cap.get(cv2.CAP_PROP_FPS) or 1.0
width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height}, {fps:.2f} fps, {total_frames} frames\n")

out_video = 'output.mp4'
writer = cv2.VideoWriter(out_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

rows = []
frame_idx = 0
t0 = time.time()

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    pil_img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

    # ---- YOLO seg ----
    yolo_results = yolo_model.predict(frame_bgr, verbose=False)[0]
    mask, yolo_cls, yolo_conf, mask_area = get_best_trail_mask(yolo_results)

    # ---- Dilate + CLIP region ----
    if mask is not None:
        mask = dilate_mask(mask, (height, width), px=MASK_DILATE_PX)
        clip_input = masked_crop_for_clip(pil_img, mask)
        used_fallback = False
    else:
        clip_input = fallback_bottom_center_crop(pil_img)
        used_fallback = True

    # ---- CLIP ----
    clip_label, clip_conf, all_scores = clip_classify_surface(clip_input)

    # ---- Annotate frame ----
    annotated = frame_bgr if mask is None else overlay_mask_bgr(frame_bgr, mask)
    label_text = f"{clip_label} ({clip_conf:.0f}%)"
    if used_fallback:
        label_text += "  [no YOLO mask]"
    cv2.putText(annotated, label_text, (12, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(annotated, label_text, (12, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
    writer.write(annotated)

    # ---- Record row ----
    row = {
        "frame": frame_idx,
        "t_sec": frame_idx / fps,
        "yolo_class": yolo_cls,
        "yolo_conf": round(yolo_conf, 3) if yolo_conf is not None else None,
        "yolo_mask_area_px": int(mask.sum()) if mask is not None else 0,
        "clip_label": clip_label,
        "clip_conf_pct": round(clip_conf, 1),
        "used_fallback": used_fallback,
    }
    for cat, score in all_scores.items():
        row[f"score_{cat}"] = round(score, 4)
    rows.append(row)

    frame_idx += 1
    if frame_idx % PROGRESS_EVERY == 0:
        elapsed = time.time() - t0
        rate = frame_idx / elapsed if elapsed > 0 else 0
        eta = (total_frames - frame_idx) / rate if rate > 0 else 0
        print(f"  {frame_idx}/{total_frames} frames  ({rate:.1f} fps, ETA {eta:.0f}s)")

cap.release()
writer.release()

per_frame_df = pd.DataFrame(rows)
per_frame_df.to_csv('surface_per_frame.csv', index=False)
print(f"\nDone in {time.time() - t0:.1f}s — {frame_idx} frames")
print(f"Wrote {out_video} and surface_per_frame.csv")

## Step 7: Per-frame summary & timeline plot

In [ ]:
import matplotlib.pyplot as plt

label_counts = per_frame_df['clip_label'].value_counts()
print("Frames per CLIP label:")
for lbl, n in label_counts.items():
    pct = n / len(per_frame_df) * 100
    print(f"  {lbl:12s}  {n:5d}  ({pct:5.1f}%)")

fb = per_frame_df['used_fallback'].sum()
print(f"\nFrames with no YOLO mask (used fallback crop): {fb} / {len(per_frame_df)}")

score_cols = [c for c in per_frame_df.columns if c.startswith('score_')]
fig, ax = plt.subplots(figsize=(14, 5))
for c in score_cols:
    ax.plot(per_frame_df['t_sec'], per_frame_df[c] * 100,
            label=c.replace('score_', ''), alpha=0.8, linewidth=1.2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('CLIP score (%)')
ax.set_title('Per-frame CLIP surface scores (raw, unsmoothed)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

per_frame_df.head(10)

## Step 8: Download outputs

In [ ]:
files.download('output.mp4')
files.download('surface_per_frame.csv')